# Final empirical synthesis
This project asks whether time-aware U.S. equity-market regimes can be identified (RQ1), and whether abnormal volume marks or predicts their transitions (RQ2). Association and prediction are evaluated separately; neither establishes causation.

SPY observations through 2017 form training data and 2018–2026 form an untouched test period. The K-means solution is a non-temporal baseline. The four-state diagonal HMM is primary because it models persistence and transition dynamics; predictive states are forward-filtered, not retrospectively smoothed.

In [1]:
from pathlib import Path
import pandas as pd, json
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FINAL = ROOT/'outputs/final_synthesis'
tables = {p.stem: pd.read_csv(p) for p in sorted((FINAL/'final_tables').glob('*.csv'))}
summary = json.loads((FINAL/'final_results_summary.json').read_text())
list(tables), summary['reproducibility']


(['table_01_sample_feature_summary',
  'table_02_kmeans_baseline',
  'table_03_hmm_specification',
  'table_04_regime_profiles',
  'table_05_hmm_transition_matrix',
  'table_06_volume_association',
  'table_07_prediction_comparison',
  'table_08_incremental_comparison',
  'table_09_robustness_secondary',
  'table_10_research_question_decisions'],
 {'models_refit': False,
  'deterministic': True,
  'accepted_outputs_modified': False})

## RQ1: Regime identification
The accepted HMM identifies four economically interpretable states—Calm Growth, Normal, Elevated Risk, and Acute Stress—which differ in return, conditional volatility, and drawdown. K-means remains a useful matched-feature baseline, but does not represent temporal persistence.

In [2]:
tables['table_03_hmm_specification'], tables['table_04_regime_profiles'], tables['table_05_hmm_transition_matrix']


(   States Covariance  Training_N         BIC  Stability_ARI_Mean  \
 0       4       diag        4277  9626.33946            0.885573   
 
    Converged_Starts  Degeneracy_Status  \
 0                20                NaN   
 
                                   Expected_Durations  
 0  Calm Growth: 32.50; Normal: 22.20; Elevated Ri...  ,
    Ordered_State    Regime_Name  count  percentage  Full_Count  \
 0              1    Calm Growth   1548   36.193594        2198   
 1              2         Normal   1306   30.535422        2061   
 2              3  Elevated Risk   1065   24.900631        1689   
 3              4   Acute Stress    358    8.370353         442   
 
    mean_Log_Return  mean_GARCH_Volatility_TrainFit  mean_Drawdown_252  \
 0         0.000998                        0.006218          -0.004950   
 1         0.000129                        0.009002          -0.028110   
 2        -0.000289                        0.013593          -0.142688   
 3        -0.001086       

## RQ2: Abnormal volume
The primary Stage 12 association is negligible and statistically insignificant. Stage 13 distinguishes training-period statistical inference from untouched-test predictive usefulness. Model 2 has a small PR-AUC gain, but slightly worse ROC AUC, essentially unchanged log loss, no material calibration improvement, and an insignificant volume coefficient and likelihood-ratio comparison. Model 3 does not rescue the general finding. These results do not imply no relationship whatsoever; they provide no stable, general, incrementally useful effect under the tested definitions. Stress- and acute-entry patterns are exploratory because results vary by transition type and some samples are small.

In [3]:
tables['table_06_volume_association'], tables['table_07_prediction_comparison'], tables['table_08_incremental_comparison']


(   Transition_Group     N      Mean    Median  CI_Lower  CI_Upper  \
 0               0.0  3526 -0.034085 -0.044758 -0.044870 -0.023301   
 1               1.0   746 -0.045449 -0.057857 -0.071044 -0.019854   
 
    Mean_Difference  Median_Difference  Hedges_g  Rank_Effect  Welch_Statistic  \
 0        -0.011364          -0.013098 -0.034207    -0.031556        -0.801926   
 1        -0.011364          -0.013098 -0.034207    -0.031556        -0.801926   
 
     P_Value  
 0  0.422782  
 1  0.422782  ,
                     Outcome  Horizon       Model   ROC_AUC    PR_AUC  \
 0  Any_Transition_Within_5D        5  Prevalence  0.500000  0.199241   
 1  Any_Transition_Within_5D        5     Model 0  0.574161  0.229296   
 2  Any_Transition_Within_5D        5     Model 1  0.675591  0.349821   
 3  Any_Transition_Within_5D        5     Model 2  0.674118  0.360357   
 4  Any_Transition_Within_5D        5     Model 3  0.671285  0.338403   
 
       Brier  Log_Loss  Precision    Recall        F1 

Statistical significance concerns sampling evidence for a coefficient; predictive usefulness concerns unseen-data accuracy; economic importance concerns whether a change is large enough to matter. A positive PR-AUC change alone is insufficient when other probability and calibration metrics are flat or worse.

In [4]:
tables['table_10_research_question_decisions']


,Research_Question,Evidence_Considered,Result,Strength_of_Evidence,Final_Interpretation
0,RQ1: Distinct regimes,Matched-feature K-means baseline; four-state d...,Supported,Strong descriptive/model-based evidence,The time-aware HMM identifies four economicall...
1,RQ2A: General contemporaneous volume association,Training five-observation transition/non-trans...,Not supported,Consistent negligible primary result,No useful general contemporaneous association ...
2,RQ2B: Conditional association,Volatility-category interaction and transition...,Exploratory only,Heterogeneous and not stable,Some cell and transition-type patterns are des...
3,RQ2C: Incremental predictive value,"Untouched-test Models 1--3, calibration, paire...",Not supported,Mixed small changes; no stable general improve...,Volume improved PR AUC slightly but not ROC AU...


## Limitations and reproducibility
Regimes are latent and model-dependent. Findings depend on the feature set, HMM specification, abnormal-volume normalization, and horizon. Daily index-level data may not generalize to intraday relationships, individual securities, or other assets. Secondary analyses are exploratory, some event counts are small, structural change may affect stability, and insignificance does not prove an exact zero. Transaction costs and a trading strategy were not evaluated.

Stage 14 reads saved canonical artifacts without refitting accepted models. The input manifest records SHA-256 hashes, and the validation report records the deterministic output checks. The empirical conclusion is therefore restrained: distinct time-aware regimes are supported, but abnormal volume does not add stable general transition-prediction value under the tested specifications.

In [5]:
json.loads((FINAL/'validation_report.json').read_text()), json.loads((FINAL/'input_manifest.json').read_text())['git_head']


({'required_inputs_valid': True,
  'input_count': 18,
  'accepted_models_refit': False,
  'tables_created': 10,
  'figures_created': 7,
  'json_finite': True,
  'deterministic_substantive_outputs': True},
 'a5b45bfb24cec1b60c47d8e803c8c29e4c9c3bc9')